In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import datetime, timedelta
import plotly.io as pio
import pytz
import plotly.graph_objects as go

In [2]:
apps_df = pd.read_csv("google_play_store_dataset.csv")
reviews_df = pd.read_csv("App_Sentiment_Analysis.csv")

In [3]:
apps_df=apps_df.dropna(subset=['Rating'])

In [4]:
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)

C:\Users\pawan\AppData\Local\Temp\ipykernel_18936\4000964054.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  apps_df[column].fillna(apps_df[column].mode()[0],inplace=True)


In [5]:
apps_df.drop_duplicates(inplace=True)
reviews_df.dropna(subset=['Translated_Review'],inplace=True)
apps_df = apps_df[apps_df['Rating']<=5]
reviews_df.dropna(subset=['Translated_Review'], inplace = True)
apps_df['Installs']= apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)
apps_df['Price'] = apps_df['Price'].str.replace('$','').astype(float)

In [6]:
merged_df = pd.merge(apps_df,reviews_df,on = 'App',how = 'inner')

In [7]:
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [8]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'K' in size:
        return float(size.replace('K',''))/1024
    else:
        return np.nan
apps_df['Size']=apps_df['Size'].apply(convert_size)

In [9]:
apps_df['Reviews'] = apps_df['Reviews'].astype(float)

In [10]:
apps_df['Log_Installs']=np.log(apps_df['Installs'])
apps_df['Log_Reviews']=np.log(apps_df['Reviews'])

In [11]:
def rating_group(rating):
    if rating >=4:
        return 'Top rated app'
    elif rating >=3:
        return 'Above Average app'
    elif rating >=2:
        return 'Average app'
    else:
        return 'Below Average app'
apps_df['Rating_Group']=apps_df['Rating'].apply(rating_group)

In [12]:
apps_df['Revenue']=apps_df['Price']*apps_df['Installs']

In [13]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,13.122363,6.874198,Above Average app,0.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0


In [14]:
sia = SentimentIntensityAnalyzer()
reviews_df['Sentiment_Score']=reviews_df['Translated_Review'].apply(lambda x : sia.polarity_scores(str(x))['compound'])

In [15]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [16]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

<IPython.core.display.Javascript object>

In [17]:
plot_containers=""

In [18]:
def save_plot_as_html(fig,filename,insight):
    global plot_containers
    filepath=os.path.join(html_files_path,filename)
    html_content=pio.to_html(fig,full_html=False,include_plotlyjs='inline')
    plot_containers+=f"""
        <div class="plot-container"id="{filename}" onclick = "openPlot('{filename}')">
            <div class="plot">{html_content}</div>
            <div class="insights">{insight}</div>
        </div>
    """
    fig.write_html(filepath,full_html=False,include_plotlyjs='inline')

In [19]:
plot_width=400
plot_height=300
plot_bg_color='black'
text_color='white'
title_font={'size':16}
axis_font={'size':12}

In [20]:
# ---- Get IST Time ----
ist = pytz.timezone("Asia/Kolkata")
ist_now = datetime.now(ist)

In [21]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,19.0,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up,9.210340,5.068904,Top rated app,0.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,13.122363,6.874198,Above Average app,0.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,8.7,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up,15.424948,11.379508,Top rated app,0.0
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,25.0,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up,17.727534,12.281384,Top rated app,0.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up,11.512925,6.874198,Top rated app,0.0


In [26]:
# ---- Time Restriction (6 PM – 9 PM IST) ----
if not (18 <= ist_now.hour < 21):
    print("Graph only visible between 6 PM and 9 PM IST")
else:

    # ---- Data Cleaning ----

    # Convert Installs & Reviews to numeric
    apps_df["Installs"] = pd.to_numeric(apps_df["Installs"], errors="coerce")
    apps_df["Reviews"] = pd.to_numeric(apps_df["Reviews"], errors="coerce")

    # Convert Date column to datetime (assuming column name is Last Updated)
    apps_df["Last Updated"] = pd.to_datetime(apps_df["Last Updated"], errors="coerce")

    # Extract Month-Year for time series
    apps_df["Month"] = apps_df["Last Updated"].dt.to_period("M").dt.to_timestamp()

    # ---- Category Translation ----
    category_translation = {
        "Beauty": "सौंदर्य",     # Hindi
        "Business": "வணிகம்",    # Tamil
        "Dating": "Dating"      # will change to German below
    }

    apps_df["Category"] = apps_df["Category"].replace(category_translation)

    # German translation
    apps_df.loc[apps_df["Category"] == "Dating", "Category"] = "Partnersuche"

    # ---- Apply Filters ----
    filtered_df = apps_df[
        (~apps_df["App"].str.lower().str.startswith(("x", "y", "z"))) &  # app name not start with x,y,z
        (~apps_df["App"].str.contains("S", case=False, na=False)) &      # no S in name
        (apps_df["Category"].str.startswith(("E", "C", "B", "स", "வ", "P"))) &  # categories starting with allowed letters
        (apps_df["Reviews"] > 500)
    ]

    # ---- Aggregate Installs Over Time ----
    grouped_df = (
        filtered_df
        .groupby(["Month", "Category"])
        .agg(Total_Installs=("Installs", "sum"))
        .reset_index()
        .sort_values("Month")
    )

    # ---- Calculate Month-over-Month Growth ----
    grouped_df["MoM_Growth"] = (
        grouped_df.groupby("Category")["Total_Installs"]
        .pct_change()
    )

    # ---- Create Plotly Figure ----
    figure = go.Figure()

    categories = grouped_df["Category"].unique()

    for cat in categories:
        cat_df = grouped_df[grouped_df["Category"] == cat]

        # Line chart
        figure.add_trace(go.Scatter(
            x=cat_df["Month"],
            y=cat_df["Total_Installs"],
            mode="lines+markers",
            name=cat
        ))

        # Highlight >20% growth
        high_growth = cat_df[cat_df["MoM_Growth"] > 0.20]

        figure.add_trace(go.Scatter(
            x=high_growth["Month"],
            y=high_growth["Total_Installs"],
            fill="tozeroy",
            mode="none",
            name=f"{cat} Growth >20%",
            opacity=0.2
        ))

    # ---- Layout Styling ----
    figure.update_layout(
        title="Monthly Trend of Total Installs by App Category",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        margin=dict(l=10, r=10, t=40, b=10)
    )

    # ---- Save HTML ----
    save_plot_as_html(
        figure,
        "TimeSeries_App_Installs_Growth.html",
        "This time-series visualization shows the trend of total installs over time segmented by app category. Areas shaded under the curve highlight periods where installs increased more than 20% month-over-month. Category names are translated into Hindi, Tamil, and German for multilingual representation. Only apps meeting strict filtering criteria are included."
    )

Graph only visible between 6 PM and 9 PM IST
